# Part 2 — External Database Configuration 💻

*Last updated:* 2026-01-08

This notebook is the step-by-step guide for creating and validating the **external YAML** files that control which **external databases**
(HGNC, MGI, UniProt, RefSeq, …) are included when IDTrack builds the identifier graph.

By the end, you will:
- have `<organism>_externals_modified.yml` files in your local repository
- understand why keeping your external database selection **small and curated** improves mapping quality
- know how to validate that your YAML is compatible with your chosen snapshot boundary

This notebook covers:
- **2.1 Human** (*Homo sapiens*) — GRCh38 (primary) + GRCh37 (legacy)
- **2.2 Mouse** (*Mus musculus*) — GRCm39
- **2.3 Pig** (*Sus scrofa*) — Sscrofa11.1
- **2.4 Adding a new organism** (advanced; may require a small code configuration step)

> **Tip:** If you're new, start with `00_idtrack_overview.ipynb` (concepts) and `01_installation_guide.ipynb` (setup).


## 2.0 — Pre-requisites (what you need before you start) 💻

- A working Python environment with IDTrack installed (`pip install idtrack`)
- Network access (first-time runs download Ensembl metadata)
- A writable **local repository** folder (IDTrack cache)

**What you get at the end:**
- `homo_sapiens_externals_modified.yml`
- `mus_musculus_externals_modified.yml`
- `sus_scrofa_externals_modified.yml`

Each file lives in your local repository and is safe to share with collaborators.


In [ ]:
# 1) Setup (run this once)
from __future__ import annotations

import os
from pathlib import Path

import yaml
import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

print('Local repository:', LOCAL_REPOSITORY)


### What is this local repository?

Think of it as your **IDTrack workspace**. It will contain:
- cached Ensembl tables (downloaded once, reused many times)
- graph snapshot files (`graph_<organism>_... .pickle`)
- your external YAML files (`*_externals_modified.yml`)

If you work on multiple projects, you can use one shared local repository (big but convenient),
or separate local repositories (clean separation, easier to archive/share).


## 2. External YAML in plain language

The YAML answers the question:

> **Which external databases should IDTrack trust and include as edges in the graph?**

Ensembl knows about *many* external resources. Some are high quality and useful, some are redundant, and some
create a lot of branching (ambiguity).

A good rule of thumb:
- enable a **small, curated** set of strong databases
- avoid enabling lots of near-duplicates

### YAML structure (what you will see)

The generated template is nested like this:

```yaml
<organism>:
  gene:
    <Database Name>:
      Assembly:
        <assembly_code>:
          Ensembl release: "90,91,92,..."
          Include: false
      Database Index: 123
      Potential Synonymous: ""
```

**You mainly edit one thing:** `Include: false` → `Include: true`.


## 3. Choose your snapshot release (reproducibility knob)

When you build graphs later, you will pick a **snapshot release** (maximum Ensembl release).

For beginners, the best choice is usually:
- **the latest release** for the organism

For reproducible research projects, consider pinning:
- the Ensembl release used by your reference annotation (e.g. the one your pipeline used)
- or the release used by a published dataset you integrate


## 2.1 — Human (Homo sapiens) 💻

Human is special in one convenient way: IDTrack ships a **default external YAML** for human.

That default already contains both common human assemblies:
- `38` → **GRCh38** (primary)
- `37` → **GRCh37** (legacy)

You have two good options:

1. **Use the shipped default as a starting point** (fast, recommended for most users)
2. **Regenerate a fresh template from live Ensembl metadata** (slower, but useful if you want to refresh for a newer release)

Recommended external databases for human (high signal, widely used):
- **HGNC Symbol** (human-readable gene symbols)
- **EntrezGene**
- **UniProtKB**
- **RefSeq_mRNA** (optionally also RefSeq proteins if your workflow needs them)

> **Tip:** Keep your allowlist small at first. You can expand later once you understand how ambiguity shows up in your results.


In [ ]:
# 4.1 Resolve organism name + latest release
organism_hs, latest_release_hs = api.resolve_organism('human')
SNAPSHOT_RELEASE_HS = latest_release_hs  # change if you want to pin
organism_hs, SNAPSHOT_RELEASE_HS


### 2.1.1 Create a DatabaseManager snapshot

This object is responsible for talking to Ensembl (downloads + caching) **bounded by your snapshot release**.


In [ ]:
dm_hs = api.get_database_manager(organism_name=organism_hs, snapshot_release=SNAPSHOT_RELEASE_HS)
dm_hs


### 2.1.2 Generate a template YAML (optional for human, required for mouse/pig)

This step can take time, because IDTrack enumerates database metadata across releases.
You typically do it once per organism (and then keep the YAML).


In [ ]:
# NOTE: This can take a while on first run (network + caching).
df_hs = dm_hs.create_database_content(just_download=False)
dm_hs.external_inst.create_template_yaml(df_hs)

template_hs = Path(dm_hs.external_inst.file_name_template_yaml())
modified_hs = Path(dm_hs.external_inst.file_name_modified_yaml(mode='configured'))
template_hs, modified_hs


### 2.1.3 Edit the YAML (two options)

#### Option A — edit by hand (recommended at least once)
1. Open the template file shown above (ends with `_externals_template.yml`).
2. Search for a database you care about (e.g. `HGNC Symbol`).
3. Change `Include: false` to `Include: true` (usually for **all assemblies listed**).
4. Save as `_externals_modified.yml` in the same folder (IDTrack will look for this first).

#### Option B — programmatic toggles (great for reproducibility)
The next cell turns on a curated allowlist automatically.


In [ ]:
# Curated allowlist for HUMAN (edit to match your lab/project policy)
allowlist_hs = [
    'HGNC Symbol',
    'EntrezGene',
    'UniProtKB',
    'RefSeq_mRNA',
]

y = yaml.safe_load(template_hs.read_text(encoding='utf-8'))
the_form = list(y[organism_hs].keys())[0]

# Turn on Include=True wherever the database exists in the template
enabled = []
for db_name in allowlist_hs:
    if db_name not in y[organism_hs][the_form]:
        continue
    for _asm, attrs in y[organism_hs][the_form][db_name]['Assembly'].items():
        attrs['Include'] = True
    enabled.append(db_name)

modified_hs.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')

print('Enabled (found in template):', enabled)
print('Wrote:', modified_hs)


### 2.1.4 Validate + preview your selections

This load step is important: it confirms your YAML contains your snapshot release and that IDTrack can parse it.


In [ ]:
_ = dm_hs.external_inst.load_modified_yaml()
print('Enabled DBs:', dm_hs.external_inst.give_list_for_case('db'))
print('Assemblies in play:', dm_hs.external_inst.give_list_for_case('assembly'))


## 2.2 — Mouse (Mus musculus) 💻

Mouse does not ship with a default external YAML in the package, so the usual workflow is:

1. generate a template YAML (from Ensembl metadata)
2. enable a curated allowlist of databases
3. validate the YAML

Mouse-specific, commonly useful databases:
- **MGI Symbol** (mouse gene symbols)
- **EntrezGene**
- **UniProtKB**
- **RefSeq_mRNA**

> **Expected output:** You will create `mus_musculus_externals_modified.yml` in your local repository.


In [ ]:
# 5.1 Resolve organism name + latest release
organism_mm, latest_release_mm = api.resolve_organism('mus musculus')
SNAPSHOT_RELEASE_MM = latest_release_mm
organism_mm, SNAPSHOT_RELEASE_MM


In [ ]:
# 5.2 Create DatabaseManager snapshot
dm_mm = api.get_database_manager(organism_name=organism_mm, snapshot_release=SNAPSHOT_RELEASE_MM)
dm_mm


In [ ]:
# 5.3 Generate template YAML
df_mm = dm_mm.create_database_content(just_download=False)
dm_mm.external_inst.create_template_yaml(df_mm)

template_mm = Path(dm_mm.external_inst.file_name_template_yaml())
modified_mm = Path(dm_mm.external_inst.file_name_modified_yaml(mode='configured'))
template_mm, modified_mm


### 2.2.1 Programmatic allowlist (mouse)

Mouse gene symbols come from **MGI** (Mouse Genome Informatics), so a typical set includes `MGI Symbol`.


In [ ]:
allowlist_mm = [
    'MGI Symbol',
    'EntrezGene',
    'UniProtKB',
    'RefSeq_mRNA',
]

y = yaml.safe_load(template_mm.read_text(encoding='utf-8'))
the_form = list(y[organism_mm].keys())[0]

enabled = []
for db_name in allowlist_mm:
    if db_name not in y[organism_mm][the_form]:
        continue
    for _asm, attrs in y[organism_mm][the_form][db_name]['Assembly'].items():
        attrs['Include'] = True
    enabled.append(db_name)

modified_mm.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('Enabled (found in template):', enabled)
print('Wrote:', modified_mm)


In [ ]:
_ = dm_mm.external_inst.load_modified_yaml()
print('Enabled DBs:', dm_mm.external_inst.give_list_for_case('db'))
print('Assemblies in play:', dm_mm.external_inst.give_list_for_case('assembly'))


## 2.3 — Pig (Sus scrofa) 💻

Pig also requires generating a template YAML and then enabling a curated allowlist.

Pig datasets often mix different naming conventions, so enabling a small set of high-signal externals is especially important.

Commonly useful databases for pig:
- **EntrezGene**
- **UniProtKB**
- **RefSeq_mRNA**

> **Expected output:** You will create `sus_scrofa_externals_modified.yml` in your local repository.


In [ ]:
# 6.1 Resolve organism name + latest release
organism_ss, latest_release_ss = api.resolve_organism('sus scrofa')
SNAPSHOT_RELEASE_SS = latest_release_ss
organism_ss, SNAPSHOT_RELEASE_SS


In [ ]:
# 6.2 Create DatabaseManager snapshot
dm_ss = api.get_database_manager(organism_name=organism_ss, snapshot_release=SNAPSHOT_RELEASE_SS)
dm_ss


In [ ]:
# 6.3 Generate template YAML
df_ss = dm_ss.create_database_content(just_download=False)
dm_ss.external_inst.create_template_yaml(df_ss)

template_ss = Path(dm_ss.external_inst.file_name_template_yaml())
modified_ss = Path(dm_ss.external_inst.file_name_modified_yaml(mode='configured'))
template_ss, modified_ss


### 2.3.1 Programmatic allowlist (pig)

Pig symbol databases can vary across releases. A safe starter set often includes `EntrezGene` and `UniProtKB`.
Use the template to inspect what is available for your snapshot release.


In [ ]:
allowlist_ss = [
    'EntrezGene',
    'UniProtKB',
    'RefSeq_mRNA',
]

y = yaml.safe_load(template_ss.read_text(encoding='utf-8'))
the_form = list(y[organism_ss].keys())[0]

enabled = []
for db_name in allowlist_ss:
    if db_name not in y[organism_ss][the_form]:
        continue
    for _asm, attrs in y[organism_ss][the_form][db_name]['Assembly'].items():
        attrs['Include'] = True
    enabled.append(db_name)

modified_ss.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('Enabled (found in template):', enabled)
print('Wrote:', modified_ss)


In [ ]:
_ = dm_ss.external_inst.load_modified_yaml()
print('Enabled DBs:', dm_ss.external_inst.give_list_for_case('db'))
print('Assemblies in play:', dm_ss.external_inst.give_list_for_case('assembly'))


## 2.4 — Adding a New Organism (Advanced) 🛠️

IDTrack currently ships with built-in support for a small set of organisms (human/mouse/pig). If you want to use a different
Ensembl-supported species, there are **two layers** to set up:

1. **Connectivity (required):** IDTrack must know which Ensembl MySQL ports host your species and which numeric **assembly code**
   appears in schema names like `<organism>_core_<release>_<assembly>`.
2. **External YAML (required):** once connectivity exists, you generate a template YAML and curate an allowlist exactly like
   mouse/pig above.

Practical recipe:

- Step A: Resolve the canonical Ensembl species name (snake_case) with `api.resolve_organism(...)`.
- Step B: Configure the species/assembly in `idtrack/_db.py` by extending `DB.assembly_mysqlport_priority`.
- Step C: Re-run the YAML-template generation workflow (`DatabaseManager.create_database_content(...)` → `create_template_yaml(...)`).
- Step D: Build a graph snapshot (Part 3) and run the sanity checks.

> **Warning:** Until Step B is done, `DatabaseManager` will raise `NotImplementedError` for that organism.


In [ ]:
# Minimal helper cell: check if an organism is supported by *this* IDTrack version.
# (Safe to run; it will not modify your installation.)

from idtrack._db import DB

print('Supported organisms in this IDTrack version:')
print('  ' + ', '.join(DB.supported_organisms))

# Example: resolve an organism name via Ensembl REST (works even if the organism isn't configured locally)
organism_query = 'danio rerio'  # zebrafish (example)
formal_name, latest_release = api.resolve_organism(organism_query)
print('Resolved via Ensembl REST ->', organism_query, '→', formal_name, '(latest release:', latest_release, ')')

if formal_name not in DB.supported_organisms:
    print()
    print('This organism is not yet configured in this IDTrack version.')
    print('To add it: extend DB.assembly_mysqlport_priority in idtrack/_db.py with its assembly code + port list,')
    print('then regenerate the external YAML using the same workflow as mouse/pig above.')
else:
    print()
    print('This organism is already configured. You can now generate a template YAML and continue.')


## 2.5 — Final checklist (before you build graphs) 💻

You should now have these three files in your local repository:
- `homo_sapiens_externals_modified.yml`
- `mus_musculus_externals_modified.yml`
- `sus_scrofa_externals_modified.yml`

The next notebook (`initialization_graph.ipynb`) will build graphs using these configs.


In [ ]:
# Quick existence check
expected = [
    LOCAL_REPOSITORY / 'homo_sapiens_externals_modified.yml',
    LOCAL_REPOSITORY / 'mus_musculus_externals_modified.yml',
    LOCAL_REPOSITORY / 'sus_scrofa_externals_modified.yml',
]
for p in expected:
    print('OK' if p.exists() else 'MISSING', '-', p)


## 2.6 — Best practices (once you get comfortable) 💡

1. **Keep allowlists small**: enabling many overlapping databases often increases ambiguity.
2. **Multiple profiles**: you can maintain different YAMLs per project (e.g. ‘strict’ vs ‘broad’).
3. **Assembly awareness**: in Ensembl database names, the last number encodes the genome assembly
   (e.g. human 38 = GRCh38; mouse 39 = GRCm39; pig 111 = Sscrofa11.1).
4. **Re-running**: you can regenerate templates after a new Ensembl release and re-apply your allowlist.
